# Factorial grid

The notebook that ran the factorial grid of the paper (Section 4, Appendix M.1, Table 19).
It was run on Colab A100 runtimes with lm-evaluation-harness 0.4.11 and vLLM. The code
cells are the ones that produced `results/controlled_eval_results.jsonl`; their outputs
are removed.

## Design

| | GSM8K | BBH | MMLU |
|---|---|---|---|
| Models | 4 | 4 | 4 |
| Prompt formats | plain, instruct, cot | plain, instruct, cot | plain, instruct, cot |
| Shot counts | 0, 5 | 0, 3 | 0, 5 |
| Temperatures | 0.0, 0.7 | 0.0, 0.7 | 0.0 |
| Seeds | 42, 123, 7 | 42, 123, 7 | 42 |
| Runs | 144 | 144 | 24 |

312 runs in total, each on 200 problems (`limit=200`; for BBH and MMLU the limit applies
to every subtask). The three prompt formats differ only in the system message, set in
`FORMAT_TEMPLATES` in Cell 3 and passed to lm-eval as `system_instruction` with
`apply_chat_template=True`.

## Running it

1. Cells 1 to 4 install the packages, set the Hugging Face token and define the grid.
   Mistral-7B-Instruct-v0.3 and Llama-3.1-8B-Instruct are gated: put a token with access
   in `HF_TOKEN` before Cell 2.
2. Cell 7a is a two-minute check that the three prompt formats give different scores.
   Run it before the grid.
3. Cell 6 runs the grid, one model after another. It appends one line per run to
   `controlled_eval_results.jsonl` in `RESULTS_DIR` and skips the runs already there, so
   it can be interrupted and started again. On a 40 GB A100 one session does not cover
   all four models: restart the runtime and run Cells 1 to 4 and 6 again.
4. Cell 7 prints how many runs are done.

Copy the results file to `experiments/controlled_eval/results/` and run the analysis
scripts listed in `experiments/README.md`.

## The MMLU runs

Six of the 24 MMLU runs failed under vLLM. The MMLU runs of three models
(Mistral-7B, Qwen2.5-7B, Qwen2.5-14B; 18 runs) were then made on the `hf` backend with
`scripts/rerun_mmlu_hf.py`, which uses the same task, prompt formats, shot counts, limit
and seed. In `results/controlled_eval_results.jsonl` those 18 lines carry
`"backend": "hf"`; the other 294 lines, including the six MMLU runs of Llama-3.1-8B, come
from this notebook. MMLU is scored by log-likelihood, and the cells that ran on both
backends agree within 0.5 pp.

In [ ]:
# Cell 1 — Install dependencies & mount Google Drive
!pip install 'lm-eval>=0.4.11' 'vllm>=0.6.3' accelerate transformers datasets \
             sentencepiece protobuf statsmodels scipy matplotlib -q

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {name}  |  Free: {free/1e9:.1f} GB / Total: {total/1e9:.1f} GB')
    if free / total < 0.9:
        print('WARNING: GPU memory is not fully free. Restart runtime before Cell 6.')
else:
    print('NO GPU DETECTED — this notebook will not work without one.')

In [ ]:
!pip install ray

In [ ]:
# Cell 2 — HuggingFace token (needed for gated Llama-3.1)
import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # e.g. 'hf_xxxx'
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print('HF token set.')
else:
    print('No HF token — Llama-3.1-8B will be skipped if gated.')

In [ ]:
# Cell 3 — Configuration (v4: extended sensitivity design)
from pathlib import Path
from itertools import product

QUICK_MODE   = False
RESULTS_DIR  = Path('/content/drive/MyDrive/eee_eval_results_v4')

# ── Model list ────────────────────────────────────────────────────────────
# All models run sequentially in Cell 6 — no commenting needed.
# The loop loads one model at a time and unloads between models.
MODELS = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    'Qwen/Qwen2.5-7B-Instruct',
    'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen/Qwen2.5-14B-Instruct',           # larger model (reviewer request)
]

# ── Grid ──────────────────────────────────────────────────────────────────────
# MMLU: loglikelihood — temperature is redundant by construction.
# GSM8K: generative — all three axes can matter.
# BBH:  generative (free-form CoT extraction) — diverse reasoning tasks.
#        Standard few-shot is 3 (not 5), so BBH gets N_SHOTS_BBH.
BENCHMARKS     = ['gsm8k', 'bbh', 'mmlu']
N_SHOTS        = [0, 5]
N_SHOTS_BBH    = [0, 3]       # BBH standard few-shot is 3

# Generative benchmarks (used by build_run_plan and Cell 6 for gen_kwargs)
GENERATIVE_BENCHMARKS = {'gsm8k', 'bbh', 'humaneval'}

# Temperature varies only for generative benchmarks
TEMPS_GENERATIVE     = [0.0, 0.7]
TEMPS_LOGLIKELIHOOD  = [0.0]          # others are mathematically redundant

# Seeds: generative runs need replication; loglikelihood is deterministic.
SEEDS_GENERATIVE     = [42, 123, 7]
SEEDS_LOGLIKELIHOOD  = [42]

# ── Prompt formats (v3+: ACTUALLY different) ─────────────────────────────────
FORMAT_TEMPLATES = {
    'plain':    None,
    'instruct': 'You are an expert problem solver. Answer each question carefully and directly.',
    'cot':      'You are an expert problem solver. Before answering, think step by step and show your reasoning. Then state the final answer.',
}
PROMPT_FORMATS = list(FORMAT_TEMPLATES.keys())

LIMIT      = 100 if QUICK_MODE else 200    # samples per benchmark per run

# ── vLLM settings ────────────────────────────────────────────────────────────
# Per-model memory utilisation: 14B needs more headroom than 7B.
VLLM_DTYPE         = 'bfloat16'
VLLM_GPU_MEM_UTIL  = {
    'default': 0.80,
    'Qwen/Qwen2.5-14B-Instruct': 0.90,    # ~28 GB weights; needs 0.90 on 40 GB A100
}
VLLM_MAX_MODEL_LEN = 2048
VLLM_ENFORCE_EAGER = True

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = RESULTS_DIR / 'controlled_eval_results.jsonl'
LOG_FILE     = RESULTS_DIR / 'run_log.txt'

def get_gpu_mem_util(model_id):
    return VLLM_GPU_MEM_UTIL.get(model_id, VLLM_GPU_MEM_UTIL['default'])

# ── Plan the run space ───────────────────────────────────────────────────
def build_run_plan():
    plan = []
    for m in MODELS:
        for bench in BENCHMARKS:
            temps = TEMPS_GENERATIVE if bench in GENERATIVE_BENCHMARKS else TEMPS_LOGLIKELIHOOD
            seeds = SEEDS_GENERATIVE if bench in GENERATIVE_BENCHMARKS else SEEDS_LOGLIKELIHOOD
            nshots = N_SHOTS_BBH if bench == 'bbh' else N_SHOTS
            for t in temps:
                for f in PROMPT_FORMATS:
                    for n in nshots:
                        for s in seeds:
                            plan.append((m, bench, t, f, n, s))
    return plan

RUN_PLAN = build_run_plan()
n_planned = len(RUN_PLAN)

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Models: {len(MODELS)} | Benchmarks: {BENCHMARKS} | Formats: {PROMPT_FORMATS}')
print(f'  Generative: temps={TEMPS_GENERATIVE} seeds={SEEDS_GENERATIVE}')
print(f'  Loglikelihood: temps={TEMPS_LOGLIKELIHOOD} seeds={SEEDS_LOGLIKELIHOOD}')
print(f'Planned runs: {n_planned}')
print(f'  → per model: {n_planned // max(len(MODELS),1)}')
for m in MODELS:
    mu = get_gpu_mem_util(m)
    print(f'  {m.split("/")[-1]:40s} gpu_mem_util={mu}')
print(f'vLLM: dtype={VLLM_DTYPE} max_len={VLLM_MAX_MODEL_LEN} eager={VLLM_ENFORCE_EAGER}')
print(f'Results dir: {RESULTS_FILE}')

In [ ]:
# Cell 4 — Helper functions
import json, time
from datetime import datetime, timezone
from collections import defaultdict

COMPLETED_KEY_FIELDS = ('model_id', 'benchmark', 'temperature', 'prompt_format', 'n_shot', 'random_seed')

def record_key(rec):
    return tuple(rec[f] for f in COMPLETED_KEY_FIELDS)

def load_completed():
    done = set()
    if not RESULTS_FILE.exists(): return done
    for line in RESULTS_FILE.read_text(encoding='utf-8').splitlines():
        if not line.strip(): continue
        try:
            r = json.loads(line)
            done.add(record_key(r))
        except Exception: pass
    return done

def append_result(rec):
    with RESULTS_FILE.open('a', encoding='utf-8') as f:
        f.write(json.dumps(rec) + '\n')

def log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with LOG_FILE.open('a', encoding='utf-8') as f:
        f.write(line + '\n')

def is_valid(benchmark, n_shot):
    if benchmark == 'humaneval' and n_shot > 0:
        return False, f'n_shot={n_shot} invalid for humaneval'
    return True, ''

def extract_score(results, benchmark):
    """Extract the primary accuracy metric from lm-eval results.

    For task groups like BBH (27 subtasks), lm-eval returns per-subtask
    results plus a group-level aggregate.  We look for:
      1. An exact match on the benchmark name (the aggregate key).
      2. Preferred metric keys: exact_match, acc, acc_norm.
      3. Fallback: first numeric non-stderr metric.

    For single tasks (gsm8k, mmlu) the old logic still works — the
    aggregate key IS the only key.
    """
    res = results.get('results', {})

    # Preferred metric names in priority order
    preferred = ('exact_match', 'acc', 'acc_norm')

    # First pass: look for exact benchmark name match (the group aggregate)
    for task_name, metrics in res.items():
        if task_name.lower().replace('_', '') == benchmark.lower().replace('_', ''):
            for k in preferred:
                if k in metrics and isinstance(metrics[k], (int, float)):
                    return round(float(metrics[k]) * 100.0, 4)
            # fallback: first numeric non-stderr
            for k, v in metrics.items():
                if isinstance(v, (int, float)) and not k.endswith(',stderr'):
                    return round(float(v) * 100.0, 4)

    # Second pass: partial match (e.g. "gsm8k" matches "leaderboard_gsm8k")
    for task_name, metrics in res.items():
        norm = task_name.lower().replace('_', '')
        if norm.startswith(benchmark.lower().replace('_', '')):
            for k in preferred:
                if k in metrics and isinstance(metrics[k], (int, float)):
                    return round(float(metrics[k]) * 100.0, 4)
            for k, v in metrics.items():
                if isinstance(v, (int, float)) and not k.endswith(',stderr'):
                    return round(float(v) * 100.0, 4)

    return None

print('Helpers loaded.')

In [ ]:
# Cell 7a — SANITY CHECK: do formats actually produce different scores?
# Run this BEFORE Cell 6. Takes ~2-3 minutes on A100.
# Loads the FIRST model, runs GSM8K at n_shot=0, temp=0.0, LIMIT=50 for each
# of the 3 formats. Prints the three scores.

import torch, lm_eval
from lm_eval import simple_evaluate
from lm_eval.models.vllm_causallms import VLLM

SANITY_MODEL = MODELS[0]
SANITY_BENCH = 'gsm8k'
SANITY_LIMIT = 50

print(f'Loading {SANITY_MODEL.split("/")[-1]} for sanity check...')
vllm_kwargs = dict(
    pretrained=SANITY_MODEL,
    dtype=VLLM_DTYPE,
    gpu_memory_utilization=get_gpu_mem_util(SANITY_MODEL),
    max_model_len=VLLM_MAX_MODEL_LEN,
    enforce_eager=VLLM_ENFORCE_EAGER,
    batch_size='auto',
)
if 'mistral' in SANITY_MODEL.lower():
    vllm_kwargs['tokenizer'] = SANITY_MODEL
    vllm_kwargs['tokenizer_mode'] = 'slow'

lm = VLLM(**vllm_kwargs)

scores = {}
for fmt, sys_inst in FORMAT_TEMPLATES.items():
    print(f'\n--- Format: {fmt} ---')
    print(f'  system_instruction: {sys_inst!r}')
    res = simple_evaluate(
        model=lm,
        tasks=[SANITY_BENCH],
        num_fewshot=0,
        gen_kwargs='seed=42,do_sample=False',
        limit=SANITY_LIMIT,
        system_instruction=sys_inst,
        apply_chat_template=True,
        fewshot_as_multiturn=False,
        random_seed=42,
        numpy_random_seed=42,
        torch_random_seed=42,
        write_out=False, log_samples=False, verbosity='ERROR',
    )
    score = extract_score(res, SANITY_BENCH)
    scores[fmt] = score
    print(f'  score = {score}')

# Unload
import gc
del lm
gc.collect()
torch.cuda.empty_cache()
try:
    from vllm.distributed.parallel_state import destroy_model_parallel
    destroy_model_parallel()
except Exception: pass

print('\n' + '=' * 60)
print('SANITY CHECK RESULT')
print('=' * 60)
for fmt, s in scores.items():
    print(f'  {fmt:10s}: {s}')

unique = len(set(scores.values()))
if unique == 1:
    print('\n*** FAIL: all three formats gave IDENTICAL scores.')
    print('    The format templates are not reaching the model.')
    print('    STOP before running Cell 6.')
elif unique == 2:
    print('\n*** PARTIAL: two formats collapse. Investigate which and why.')
else:
    print('\n    OK: all three formats produce distinct scores. Proceed to Cell 6.')
    max_spread = max(scores.values()) - min(scores.values())
    print(f'    Score spread across formats: {max_spread:.2f} pp')

In [ ]:
# Cell 6 — Run experiment (safe to interrupt/resume)
# All models run sequentially — no commenting/uncommenting needed.
import gc
import torch
import lm_eval
from lm_eval import simple_evaluate
from lm_eval.models.vllm_causallms import VLLM
from transformers import AutoConfig

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (no GPU!)'
print(f'lm_eval {lm_eval.__version__}  |  {gpu}')

completed = load_completed()
print(f'Resuming from {len(completed)} completed cells')

# Group the run plan by model so we only load each model once
from collections import defaultdict
plan_by_model = defaultdict(list)
for cell in RUN_PLAN:
    plan_by_model[cell[0]].append(cell)

run_n = errors = 0
t0_global = time.time()
total = len(RUN_PLAN)

for model_id, cells in plan_by_model.items():
    short = model_id.split('/')[-1]
    try:
        AutoConfig.from_pretrained(model_id)
    except Exception as e:
        log(f'SKIP {short}: {e}')
        continue

    remaining_cells = [c for c in cells if c not in completed
                       and is_valid(c[1], c[4])[0]]
    if not remaining_cells:
        log(f'=== {short}: all cells complete, skipping model load ===')
        run_n += len(cells)
        continue

    mem_util = get_gpu_mem_util(model_id)
    log(f'=== Loading {short} with vLLM ({len(remaining_cells)} cells remaining, mem_util={mem_util}) ===')
    t_load = time.time()

    vllm_kwargs = dict(
        pretrained=model_id,
        dtype=VLLM_DTYPE,
        gpu_memory_utilization=mem_util,
        max_model_len=VLLM_MAX_MODEL_LEN,
        enforce_eager=VLLM_ENFORCE_EAGER,
        batch_size='auto',
    )
    # Mistral v0.3+: force slow tokenizer to avoid mistral_common breakage
    if 'mistral' in model_id.lower():
        vllm_kwargs['tokenizer'] = model_id
        vllm_kwargs['tokenizer_mode'] = 'slow'

    try:
        lm = VLLM(**vllm_kwargs)
    except Exception as e:
        log(f'FAILED to load {short}: {e}')
        continue
    log(f'  Loaded in {round(time.time() - t_load, 1)}s')

    for cell in cells:
        _, benchmark, temperature, prompt_format, n_shot, seed = cell
        run_n += 1

        base_rec = dict(
            model_id=model_id, benchmark=benchmark,
            temperature=temperature, prompt_format=prompt_format,
            n_shot=n_shot, random_seed=seed, score=None,
            timestamp=datetime.now(timezone.utc).isoformat(),
            eval_library_version=lm_eval.__version__,
            limit=LIMIT,
        )

        valid, reason = is_valid(benchmark, n_shot)
        if not valid:
            append_result({**base_rec, 'status': 'invalid_config', 'reason': reason})
            continue

        if cell in completed:
            continue

        log(f'[{run_n}/{total}] {short}/{benchmark} temp={temperature} fmt={prompt_format} n={n_shot} seed={seed}')

        # ── Build gen_kwargs ──────────────────────────────────────────────────
        gen_kwargs = f'seed={seed}'
        if benchmark in GENERATIVE_BENCHMARKS:
            if temperature > 0.0:
                gen_kwargs += f',temperature={temperature},do_sample=True'
            else:
                gen_kwargs += ',do_sample=False'

        # ── System instruction for this format ───────────────────────────────
        sys_inst = FORMAT_TEMPLATES[prompt_format]

        t0 = time.time()
        try:
            results = simple_evaluate(
                model=lm,
                tasks=[benchmark],
                num_fewshot=n_shot,
                gen_kwargs=gen_kwargs,
                limit=LIMIT,
                system_instruction=sys_inst,
                apply_chat_template=True,
                fewshot_as_multiturn=False,
                random_seed=seed,
                numpy_random_seed=seed,
                torch_random_seed=seed,
                write_out=False, log_samples=False, verbosity='ERROR',
            )
            score = extract_score(results, benchmark)
            elapsed = round(time.time() - t0, 1)
            status = 'ok' if score is not None else 'score_extraction_failed'
            log(f'  -> score={score} ({elapsed}s)')
            append_result({**base_rec, 'score': score, 'status': status,
                           'reason': '' if score else 'extraction failed',
                           'elapsed_sec': elapsed})
            completed.add(cell)
        except Exception as exc:
            errors += 1
            log(f'  ERROR: {exc}')
            append_result({**base_rec, 'status': 'error', 'reason': str(exc)})

    log(f'=== Unloading {short} ===')
    try: del lm
    except NameError: pass
    gc.collect()
    torch.cuda.empty_cache()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel
        destroy_model_parallel()
    except Exception: pass
    time.sleep(5)   # let GPU memory settle before next model

elapsed_min = round((time.time() - t0_global) / 60, 1)
log(f'=== Finished. {len(completed)} done, {errors} errors, {elapsed_min} min total ===')

In [ ]:
# Cell 7 — Progress check (run any time, independent of Cell 6)
import json
from collections import defaultdict

if not RESULTS_FILE.exists():
    print('No results yet.')
else:
    recs = []
    for line in RESULTS_FILE.read_text(encoding='utf-8').splitlines():
        if line.strip():
            try: recs.append(json.loads(line))
            except Exception: pass
    ok = [r for r in recs if r.get('status') == 'ok' and r.get('score') is not None]
    print(f'Records total: {len(recs)}  |  OK: {len(ok)}  |  '
          f'Errors: {sum(1 for r in recs if r.get("status")=="error")}')

    by_model = defaultdict(int)
    by_bench = defaultdict(int)
    by_mbf = defaultdict(int)
    for r in ok:
        by_model[r['model_id'].split('/')[-1]] += 1
        by_bench[r['benchmark']] += 1
        by_mbf[(r['model_id'].split('/')[-1], r['benchmark'], r['prompt_format'])] += 1
    print(f'  By model: {dict(sorted(by_model.items()))}')
    print(f'  By bench: {dict(sorted(by_bench.items()))}')
    print(f'  By (model, bench, format) cells filled:')
    for k, n in sorted(by_mbf.items()):
        print(f'    {k}: {n} seeds')

    # Diagnostic: score spread per (model, benchmark, format) cell across seeds.
    # If the spread is 0.00 across all cells, seeds are not propagating.
    by_cell = defaultdict(list)
    for r in ok:
        k = (r['model_id'].split('/')[-1], r['benchmark'],
             r['temperature'], r['prompt_format'], r['n_shot'])
        by_cell[k].append(r['score'])
    spreads = [max(v)-min(v) for v in by_cell.values() if len(v) >= 2]
    if spreads:
        import statistics
        print(f'\n  Seed spread (max-min within each cell, pp):')
        print(f'    mean: {statistics.mean(spreads):.3f}  '
              f'median: {statistics.median(spreads):.3f}  '
              f'max: {max(spreads):.3f}')
        if max(spreads) == 0.0:
            print('    *** WARNING: no seed variation. Generative runs are not stochastic.')